# L06 · Datos de entrenamiento, balance y overlap

**S6 · Lectura: Cap. 7 · 40–55 min**

## La pregunta

¿Hay personas comparables en tratamiento y control?

## Qué harás

Usar los splits oficiales y revisar el solapamiento.

- Lee la idea breve.
- Ejecuta la demostración y mira el gráfico.
- Completa las celdas marcadas.
- Escribe qué significa el resultado y qué no permite afirmar.

**Producto:** Diagnóstico de balance, propensión y soporte común.

> **Idea clave:** Sin unidades comparables, el efecto se extrapola y pierde credibilidad.

La solución está en `la ruta de rescate docente (no incluida)`. Intenta primero el notebook del estudiante.

In [ ]:
# Localiza lab_common.py (carpeta actual o una superior)
import sys, pathlib
for _p in [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3]]:
    if (_p / "lab_common.py").exists():
        sys.path.insert(0, str(_p)); break

## 0. Preparación

In [ ]:
from lab_common import *
from lab_assumptions import smd as smd_ref
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
rng = setup()
sc = load_scenario("real/jobs"); sc.card()

In [ ]:
# Vista rápida de los datos del escenario
import matplotlib.pyplot as plt
import numpy as np
_fig, _ax = plt.subplots(1, 2, figsize=(9, 3.2))
try:
    _scenario = globals().get("sc") or globals().get("lc")
    if _scenario is None:
        raise NameError("No hay un escenario cargado para el gráfico editorial")
    _df = _scenario.train
    if _scenario.treatment is not None and _scenario.outcome is not None:
        _t = _df[_scenario.treatment].to_numpy(float)
        _y = _df[_scenario.outcome].to_numpy(float)
        if np.unique(_t).size == 2:
            _counts = [int((_t == value).sum()) for value in (0, 1)]
            _ax[0].bar(["control", "tratado"], _counts)
            _ax[0].set_title("Asignación")
            _ax[1].boxplot([_y[_t == 0], _y[_t == 1]], labels=["control", "tratado"])
            _ax[1].set_title("Resultado observado")
        else:
            _ax[0].hist(_t, bins=20)
            _ax[0].set_title(_scenario.treatment)
            _ax[1].scatter(_t, _y, s=8, alpha=.35)
            _ax[1].set_title(f"{_scenario.treatment} frente a {_scenario.outcome}")
    elif _scenario.adj is not None:
        _ax[0].imshow(_scenario.adj, cmap="Greys", interpolation="nearest")
        _ax[0].set_title("Grafo del escenario")
        _ax[1].axis("off")
    else:
        _ax[0].axis("off"); _ax[1].axis("off")
    _fig.suptitle(f"Escenario: {_scenario.name}")
    _fig.tight_layout(); plt.show()
finally:
    plt.close(_fig)

## 1. Primero: ¿de dónde sale el tratamiento? (§7.1)
Antes de cualquier número, la **ficha de procedencia del mecanismo de asignación**. Responde cuatro preguntas; de sus respuestas depende qué supuesto necesitas.

In [ ]:
ficha = pd.DataFrame([
    ("¿Quién decidió la asignación?",        "participantes e inscripción en el programa (autoselección parcial)"),
    ("¿Con qué regla?",                      sc.meta.get("treatment", "programa de capacitación laboral")),
    ("¿Cuándo se midieron las covariables?", "antes del programa (edad, educación, ingresos 1974–75); el resultado es re78"),
    ("¿Qué origen tienen los datos?",        str(sc.meta.get("data_origin", "sintético"))[:70]),
], columns=["pregunta", "respuesta para este escenario"])
show(ficha)

**Mira el resultado.** Si la asignación fue una autoselección parcial, ¿qué supuesto necesitas y de qué tipo es?

**Respuesta:**

**No confusión condicional**: que, dadas las covariables medidas, quien se inscribe no difiera en sus resultados potenciales. Es **no observable**: se declara y se acota, no se prueba con los datos.



## 2. Medir balance: la diferencia media estandarizada (§7.2, ec. 7.13)
$$\mathrm{SMD}=\frac{\bar x_1-\bar x_0}{\sqrt{(s_1^2+s_0^2)/2}}\qquad \mathrm{VR}=\frac{s_1^2}{s_0^2}$$
Criterios de Rubin: $|\mathrm{SMD}|<0{,}25$ aceptable (ideal $<0{,}1$) y $\mathrm{VR}\in[0{,}5;\,2]$.

In [ ]:
# ✏️ Completa tú — smd_vr
# Pista: Usa varianza muestral (ddof=1).
def smd_vr(x, t):
    """Devuelve (SMD, razón de varianzas) de x entre tratados (t==1) y controles (t==0)."""
    raise NotImplementedError("diferencia de medias entre √((s1²+s0²)/2); razón s1²/s0²")

In [ ]:
X, y, d = sc.xyt("train")
filas = []
for j, c in enumerate(sc.covariates):
    s_, vr = smd_vr(X[:, j], d)
    filas.append({"covariable": c, "SMD": s_, "|SMD|": abs(s_), "VR": vr,
                  "veredicto": "ok" if abs(s_) < .1 else ("aceptable" if abs(s_) < .25 else "desbalance")})
bal = pd.DataFrame(filas)
show(bal.sort_values("|SMD|", ascending=False).round(3))
lab_check("smd_vr", lambda: smd_vr(X[:, 0], d)[0], [((), smd_ref(X[:, 0], d), 1e-2)])

## 3. Por qué **no** se usa un t-test para el balance (§7.2)
Un t-test mezcla el *tamaño* del desbalance con el *tamaño de la muestra*. Fijamos un desbalance de SMD = 0,15 (construido para que sea exacto) y variamos solo $n$.

In [ ]:
filas = []
for n in (100, 400, 1600, 6400):
    r = np.random.default_rng(1); t = r.binomial(1, .5, n); z = r.normal(size=n)
    x = z.copy()                                    # cada grupo se estandariza: el SMD queda fijo en 0,15
    for g, m in ((1, .15), (0, 0.0)):
        x[t == g] = (z[t == g] - z[t == g].mean()) / z[t == g].std() + m
    s_, _ = smd_vr(x, t); p = stats.ttest_ind(x[t == 1], x[t == 0], equal_var=False).pvalue
    filas.append({"n": n, "SMD": round(s_, 3), "p-valor del t-test": round(p, 4), "t-test dice": "desbalance" if p < .05 else "balanceado"})
show(pd.DataFrame(filas))

**Mira el resultado.** El desbalance real es el mismo en las cuatro filas. ¿Qué cambia y qué debes reportar?

**Respuesta:**

Cambia el **p-valor**, no el SMD. Con $n$ pequeño el t-test «absuelve» un desbalance real y con $n$ grande «condena» uno trivial. Se reporta el **tamaño** (SMD), no la significancia.



## 4. Muchas covariables, muchas pruebas: Holm y Benjamini–Hochberg (§7.3)
Con 30 covariables **perfectamente balanceadas** (aleatorización), ¿cuántas veces "encontramos" algún desbalance por azar? Si decides con p < 0,05 sin corregir, casi siempre.

In [ ]:
# ✏️ Completa tú — holm
# Pista: Para el rango k = 0,1,...: rechaza si p_(k) ≤ α/(m−k); si no, para.
def holm(p, alpha=0.05):
    """Holm (step-down): True en los p-valores rechazados."""
    raise NotImplementedError("ordena los p; compara el k-ésimo con alpha/(m-k); detente en el primer no rechazo")

In [ ]:
def bh(p, alpha=0.05):
    p = np.asarray(p, float); m = len(p); o = np.argsort(p)
    ok = p[o] <= alpha * (np.arange(1, m + 1) / m)
    k = np.flatnonzero(ok).max() + 1 if ok.any() else 0
    rech = np.zeros(m, bool); rech[o[:k]] = True
    return rech
lab_check("holm", holm, [((np.array([.001, .02, .03, .5]),), np.array([True, False, False, False]), 0)])
r = np.random.default_rng(5); n_exp, m, n = 400, 30, 300
alguno = {"sin corregir (p<0,05)": 0, "Holm": 0, "BH": 0}
for _ in range(n_exp):
    t = r.binomial(1, .5, n); P = np.array([stats.ttest_ind(r.normal(size=n)[t == 1], r.normal(size=n)[t == 0]).pvalue for _ in range(m)])
    alguno["sin corregir (p<0,05)"] += (P < .05).any(); alguno["Holm"] += holm(P).any(); alguno["BH"] += bh(P).any()
show(pd.DataFrame({k: [v / n_exp] for k, v in alguno.items()}, index=["P(declarar algún desbalance)"]).round(3))

## 5. Experimental frente a observacional (§7.4, estilo LaLonde)
En el estudio clásico, el mismo programa se evaluó con un **grupo de control aleatorizado** y con **controles tomados de una encuesta**. Reproducimos la estructura con un mundo simulado (el benchmark no trae una columna que distinga ambos grupos): el efecto verdadero es de 1 800.

In [ ]:
def lalonde_like(rng, n_t=185, n_c_exp=260, n_c_obs=1500, efecto=1800):
    def covs(n, shift):
        edad = rng.normal(25 + shift * 8, 6, n).clip(17, 55); educ = rng.normal(10 + shift * 2, 2, n).clip(3, 16)
        ing = np.clip(rng.normal(3000 + shift * 9000, 3500, n), 0, None)
        return np.column_stack([edad, educ, ing])
    Xt = covs(n_t, 0); Xce = covs(n_c_exp, 0); Xco = covs(n_c_obs, 1)       # los controles observacionales son distintos
    f = lambda X: 4000 + 0.6 * X[:, 2] + 120 * X[:, 1] + 20 * X[:, 0]
    yt = f(Xt) + efecto + rng.normal(0, 2500, n_t); yce = f(Xce) + rng.normal(0, 2500, n_c_exp); yco = f(Xco) + rng.normal(0, 2500, n_c_obs)
    return (Xt, yt), (Xce, yce), (Xco, yco)
(Xt, yt), (Xce, yce), (Xco, yco) = lalonde_like(np.random.default_rng(11))
def tabla_smd(Xa, Xb):
    t = np.r_[np.ones(len(Xa)), np.zeros(len(Xb))]; Z = np.vstack([Xa, Xb])
    return np.array([smd_vr(Z[:, j], t)[0] for j in range(Z.shape[1])])
s_exp, s_obs = tabla_smd(Xt, Xce), tabla_smd(Xt, Xco)
show(pd.DataFrame({"covariable": ["edad", "educación", "ingreso previo"], "|SMD| exp.": np.abs(s_exp), "|SMD| obs.": np.abs(s_obs)}).round(3))
e1, se1 = diff_in_means(np.r_[yt, yce], np.r_[np.ones(len(yt)), np.zeros(len(yce))])
e2, se2 = diff_in_means(np.r_[yt, yco], np.r_[np.ones(len(yt)), np.zeros(len(yco))])
print(f"DiM con control aleatorizado : {fmt_ci(e1, se1, 0)}")
print(f"DiM con control observacional: {fmt_ci(e2, se2, 0)}   (efecto verdadero 1800)")

**Mira el resultado.** El balance observacional es malo y el sesgo, grande. ¿Ajustar por esas covariables bastaría?

**Respuesta:**

Depende de que **todo** lo que difiere esté medido. Es justo la lección de LaLonde: con controles observacionales los estimadores recuperaban el efecto solo a veces. El SMD diagnostica *lo medido*; lo no medido sigue siendo un supuesto.



## 6. Puente de A1: particiones y un primer vistazo al solapamiento
Para que A1 sea reproducible: tamaños, ausencia de solape entre particiones y una huella (SHA-256) de cada una. Y un primer ajuste del propensity **solo con train** (el diagnóstico completo de solapamiento es L10).

In [ ]:
from sklearn.linear_model import LogisticRegression
print({k: len(sc.split(k)) for k in ("train", "val", "test")})
if "row_id" in sc.train.columns:
    ids = [set(sc.split(k).row_id) for k in ("train", "val", "test")]
    print("solape entre particiones:", len(ids[0] & ids[1]) + len(ids[0] & ids[2]) + len(ids[1] & ids[2]))
print("huella train:", sha256_df(sc.train)[:23], "…")
Z = (X - X.mean(0)) / X.std(0)
ps = LogisticRegression(max_iter=1000).fit(Z, d).predict_proba(Z)[:, 1]
print(f"propensity (solo train): rango [{ps.min():.3f}, {ps.max():.3f}] · fracción en [0,05; 0,95] = {np.mean((ps > .05) & (ps < .95)):.3f}")

## 7. Supuesto y bitácora

In [ ]:
fila = bitacora(
    pregunta="¿Son comparables tratados y controles en lo medido, y qué sabemos del mecanismo de asignación?",
    metodo="Ficha de procedencia + SMD y razón de varianzas (criterios de Rubin)",
    estimando="Diagnóstico de balance (no es un efecto)",
    numero_ic=f"|SMD| máximo = {bal['|SMD|'].max():.3f}; {int((bal['|SMD|'] >= .1).sum())} de {len(bal)} covariables con |SMD| ≥ 0,1 (sin IC: es un diagnóstico)",
    supuesto=supuesto("observable", "medido", f"|SMD| máximo = {bal['|SMD|'].max():.3f}; el supuesto de fondo (no confusión) queda declarado"),
    decision="continuar" if bal["|SMD|"].max() < .25 else "alertar", lab="L06")

---
## Qué queda para el libro
Ejercicios del libro cubiertos: comparar distribuciones de SMD (experimental vs observacional) y Holm frente a p-valores crudos. El lab añade, como **puente de A1**, las particiones y un primer diagnóstico de solapamiento (el solapamiento se desarrolla en L10).

*Material formativo (sin nota). Los datos son sintéticos: ver la ficha del escenario.*